# 13. Численная устойчивость и форматы чисел

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/13_numerics.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#m13). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Стабильный softmax и log-sum-exp

In [ ]:
import numpy as np
z = np.array([1000., 1001., 1002.])
naive = np.exp(z) / np.exp(z).sum()                 # [nan nan nan] + RuntimeWarning
def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()
def log_softmax(z):
    m = z.max(); return z - m - np.log(np.exp(z - m).sum())
print(softmax(z), log_softmax(z))                   # [0.09 0.245 0.665]  [-2.41 -1.41 -0.41]

### 🏋️ Практика модуля

**13.1.** Покажите, что формула дисперсии $\mathbb{E}[x^2] - \mathbb{E}[x]^2$ ломается в FP32 для данных вокруг $10^4$ с малым разбросом.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
x = (1e4 + np.random.randn(100_000) * 0.01).astype(np.float32)
naive = (x**2).mean() - x.mean()**2
print(naive, x.var(), np.var(x.astype(np.float64)))
# naive может быть даже отрицательным; x.var() вычитает среднее заранее и даёт ~1e-4

### 🏋️ Практика модуля

**13.2.** Напишите стабильную сигмоиду, корректную для $z = \pm 1000$.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1 / (1 + np.exp(-z[pos]))
    ez = np.exp(z[~pos]); out[~pos] = ez / (1 + ez)     # exp от отрицательного — не переполняется
    return out
print(sigmoid([-1000, 0, 1000]))   # [0. 0.5 1.]